# 02q — YOLOv8s-cls Dual-Run Training (3 single-crack classes)

**Project:** UREP 32-0210-250078 | Crack Classification

**Architecture:** YOLOv8s-cls (Ultralytics).

**Classes (3):** debonding, flexural, shear.

## Two training runs in this notebook

1. **Run A — QU only** (3-class subset, materialized at `data/split_3class/`).
2. **Run B — QU + AutoCAD synthetic + mirror** (3-class, materialized at `data/split_3class_synth/` with `synth_mirror_*.jpg` flipped copies on disk).

Ultralytics owns its own dataloader and cannot accept a custom flip flag, so for YOLO the synthetic mirrors are baked to disk by `prepare_3class_yolo_split(with_synthetic=True)`. Ultralytics random aug (`fliplr=0.5`, `degrees=15.0`) still applies to all images on top — same parameters as the existing `02e_training_yolo.ipynb`.

Test evaluation for both runs uses the same clean 3-class test set from `data/split_3class/test/`.

## Setup

In [ ]:
import sys
sys.path.insert(0, "..")

import os
import numpy as np
from ultralytics import YOLO

import config
from src.dataset import prepare_dataset, split_dataset, prepare_3class_yolo_split
from src.evaluation import evaluate_predictions
from src.device import set_seed

set_seed(config.RANDOM_SEED)

CLASS_NAMES_3 = ["debonding", "flexural", "shear"]

OUT_QU      = os.path.join(config.OUTPUT_DIR, "yolo_3class_qu")
OUT_QUSYNTH = os.path.join(config.OUTPUT_DIR, "yolo_3class_qu_synth")
for d in (OUT_QU, OUT_QUSYNTH):
    os.makedirs(os.path.join(d, "plots"), exist_ok=True)

print(f"Architecture: YOLOv8s-cls (3-class)")
print(f"Classes:      {CLASS_NAMES_3}")
print(f"OUT_QU:       {OUT_QU}")
print(f"OUT_QUSYNTH:  {OUT_QUSYNTH}")

### Ensure base 6-class split exists

In [ ]:
# Ensure base 6-class split exists (idempotent)
if not os.path.exists(config.SPLIT_DIR) or not os.path.isdir(os.path.join(config.SPLIT_DIR, "train")):
    print("Preparing base dataset from QU structure...")
    prepare_dataset()
    split_dataset()
else:
    print(f"Base split exists: {config.SPLIT_DIR}")

---
# Run A — QU only

### Run A: materialize 3-class split (no synth)

In [ ]:
counts_a = prepare_3class_yolo_split(with_synthetic=False, class_names=tuple(CLASS_NAMES_3))
SPLIT_A = os.path.join(config.DATA_DIR, "split_3class")

# Per-subset file count log
def _yolo_count(split_dir, label):
    print(f"\n{'='*60}\n{label}\n{'='*60}")
    for subset in ["train", "val", "test"]:
        for c in CLASS_NAMES_3:
            d = os.path.join(split_dir, subset, c)
            n_total = len([f for f in os.listdir(d) if os.path.splitext(f)[1].lower() in {".jpg",".jpeg",".png",".bmp"}]) if os.path.isdir(d) else 0
            n_synth = len([f for f in os.listdir(d) if f.startswith("synth_") and not f.startswith("synth_mirror_")]) if os.path.isdir(d) else 0
            n_mirror = len([f for f in os.listdir(d) if f.startswith("synth_mirror_")]) if os.path.isdir(d) else 0
            n_real = n_total - n_synth - n_mirror
            print(f"  {subset:<5}/{c:<12} total={n_total:>6,}  real={n_real:>6,}  synth={n_synth:>4,}  mirror={n_mirror:>4,}")

_yolo_count(SPLIT_A, "RUN A — split_3class — file counts")

### Run A: train YOLO

In [ ]:
model_a = YOLO(config.YOLO_MODEL)
results_a = model_a.train(
    data=SPLIT_A,
    epochs=config.YOLO_EPOCHS,
    imgsz=config.YOLO_IMG_SIZE,
    batch=16,
    patience=config.YOLO_PATIENCE,
    lr0=config.YOLO_LR0,
    lrf=config.YOLO_LRF,
    dropout=config.YOLO_DROPOUT,
    optimizer="AdamW",
    degrees=15.0, fliplr=0.5, flipud=0.0, shear=0.0,
    seed=config.RANDOM_SEED,
    project=OUT_QU, name="train", exist_ok=True, verbose=True,
)

### Run A: evaluate on clean 3-class test set

In [ ]:
best_a = os.path.join(OUT_QU, "train", "weights", "best.pt")
model_a_eval = YOLO(best_a)

test_dir = os.path.join(SPLIT_A, "test")
y_true_a, y_pred_a = [], []
for cls_idx, cls_name in enumerate(CLASS_NAMES_3):
    cls_dir = os.path.join(test_dir, cls_name)
    if not os.path.isdir(cls_dir): continue
    files = [f for f in os.listdir(cls_dir) if os.path.splitext(f)[1].lower() in {".jpg",".jpeg",".png",".bmp"}]
    for fname in files:
        result = model_a_eval.predict(os.path.join(cls_dir, fname), verbose=False)
        pred_name = result[0].names[result[0].probs.top1]
        pred_idx = CLASS_NAMES_3.index(pred_name) if pred_name in CLASS_NAMES_3 else result[0].probs.top1
        y_true_a.append(cls_idx)
        y_pred_a.append(pred_idx)
    print(f"  {cls_name}: {len(files)} predictions done")

metrics_a = evaluate_predictions(
    np.array(y_true_a), np.array(y_pred_a),
    class_names=CLASS_NAMES_3,
    output_dir=OUT_QU, model_name="yolo_3c_qu",
)

---
# Run B — QU + synthetic + on-disk mirrors

### Run B: materialize 3-class split with synth + mirror

In [ ]:
counts_b = prepare_3class_yolo_split(with_synthetic=True, class_names=tuple(CLASS_NAMES_3))
SPLIT_B = os.path.join(config.DATA_DIR, "split_3class_synth")

_yolo_count(SPLIT_B, "RUN B — split_3class_synth — file counts")

# Mirror count assertion
n_mirror_total = 0
for c in CLASS_NAMES_3:
    d = os.path.join(SPLIT_B, "train", c)
    n_mirror_total += len([f for f in os.listdir(d) if f.startswith("synth_mirror_")])
print(f"\nTotal synth_mirror_ files in train: {n_mirror_total}")
assert n_mirror_total == 102 + 302 + 402, f"Expected 806 mirror files, got {n_mirror_total}"
print("OK — every synthetic image has a synth_mirror_*.jpg counterpart on disk.")

### Run B: train YOLO

In [ ]:
model_b = YOLO(config.YOLO_MODEL)
results_b = model_b.train(
    data=SPLIT_B,
    epochs=config.YOLO_EPOCHS,
    imgsz=config.YOLO_IMG_SIZE,
    batch=16,
    patience=config.YOLO_PATIENCE,
    lr0=config.YOLO_LR0,
    lrf=config.YOLO_LRF,
    dropout=config.YOLO_DROPOUT,
    optimizer="AdamW",
    degrees=15.0, fliplr=0.5, flipud=0.0, shear=0.0,
    seed=config.RANDOM_SEED,
    project=OUT_QUSYNTH, name="train", exist_ok=True, verbose=True,
)

### Run B: evaluate on the same clean 3-class test set used by Run A

In [ ]:
best_b = os.path.join(OUT_QUSYNTH, "train", "weights", "best.pt")
model_b_eval = YOLO(best_b)

# Use SPLIT_A test dir (clean, no synth/mirror) for fair comparison with Run A
test_dir = os.path.join(SPLIT_A, "test")
y_true_b, y_pred_b = [], []
for cls_idx, cls_name in enumerate(CLASS_NAMES_3):
    cls_dir = os.path.join(test_dir, cls_name)
    if not os.path.isdir(cls_dir): continue
    files = [f for f in os.listdir(cls_dir) if os.path.splitext(f)[1].lower() in {".jpg",".jpeg",".png",".bmp"}]
    for fname in files:
        result = model_b_eval.predict(os.path.join(cls_dir, fname), verbose=False)
        pred_name = result[0].names[result[0].probs.top1]
        pred_idx = CLASS_NAMES_3.index(pred_name) if pred_name in CLASS_NAMES_3 else result[0].probs.top1
        y_true_b.append(cls_idx)
        y_pred_b.append(pred_idx)
    print(f"  {cls_name}: {len(files)} predictions done")

metrics_b = evaluate_predictions(
    np.array(y_true_b), np.array(y_pred_b),
    class_names=CLASS_NAMES_3,
    output_dir=OUT_QUSYNTH, model_name="yolo_3c_qusynth",
)

---
# Comparison: Run A vs Run B

In [ ]:
print(f"\n{'='*78}")
print(f"YOLOv8s-cls — 3-CLASS COMPARISON  (test set: {len(y_true_a):,} images)")
print(f"{'='*78}")
print(f"  {'Metric':<24} {'Run A: QU only':>18} {'Run B: QU+synth+mir':>22}")
print(f"  {'-'*64}")
for key in ["accuracy", "precision_macro", "recall_macro", "f1_macro", "f1_weighted", "mean_iou"]:
    print(f"  {key:<24} {metrics_a[key]:>18.4f} {metrics_b[key]:>22.4f}")
print(f"  {'-'*64}")
print(f"  Per-class IoU:")
for cls in CLASS_NAMES_3:
    a = metrics_a["iou_per_class"][cls]
    b = metrics_b["iou_per_class"][cls]
    delta = b - a
    print(f"    {cls:<14} {a:>18.4f} {b:>22.4f}   (Δ={delta:+.4f})")
print(f"{'='*78}")